# ReplayAxis · Two-view markerless motion analysis

Run the two code cells below for a dependency-free synthetic demonstration inside Jupyter. Orbit, zoom, play, pause, scrub or step through the reconstructed skeleton. The right panel shows source frame IDs, usable joints, frame-pair skew after applying the configured start offsets, and reprojection error. Select a joint to inspect its scores, ray angle and per-camera residuals. The exported HTML opens directly in a browser. The default cyan wireframe is a procedural body proxy driven by the reconstructed joints. Use the View selector to inspect the skeleton. The wireframe is illustrative and is not a measured body surface. During review, click a reconstructed joint or use the Mark joint button. The highlight starts at the selected frame, follows the joint until unmarked, and is omitted when the joint is withheld. Rewinding restores the marking state for that frame. Marks are review annotations held in the current viewer session. The joint CSV has one row per frame and joint, with withheld positions left blank.

### Capture requirements

The prototype reconstructs **one athlete in two fixed, overlapping views of the same take**. The clips must share a time reference. Metric reconstruction requires camera calibration that matches the videos' exact resolution, crop, zoom and placement. Moving cameras, unrelated takes, severe occlusion and poor camera separation can make depth unobservable. The output is a 3D skeleton rather than a photorealistic rendering.

In [ ]:
import csv, json, math
from pathlib import Path
from html import escape
JOINTS = {0:'Head',11:'Left shoulder',12:'Right shoulder',13:'Left elbow',14:'Right elbow',
          15:'Left wrist',16:'Right wrist',23:'Left hip',24:'Right hip',
          25:'Left knee',26:'Right knee',27:'Left ankle',28:'Right ankle'}
BONES = [(0,11),(0,12),(11,12),(11,13),(13,15),(12,14),(14,16),
         (11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]
def dot(a,b):
    return sum(x*y for x,y in zip(a,b))
def diff(a,b):
    return [x-y for x,y in zip(a,b)]
def norm(a):
    return math.sqrt(dot(a,a))
def project(point,camera):
    q = [dot(row,point)+t for row,t in zip(camera['R'],camera['t'])]
    if q[2]<=0:
        return None
    u,v,w = [dot(row,q) for row in camera['K']]
    return [u/w,v/w]
def ray(pixel,camera):
    k=camera['K']
    q=[(pixel[0]-k[0][2])/k[0][0],(pixel[1]-k[1][2])/k[1][1],1]
    cols=list(zip(*camera['R']))
    center=[-dot(col,camera['t']) for col in cols]
    direction=[dot(col,q) for col in cols]
    length=norm(direction)
    return center,[x/length for x in direction]
def triangulate(a,b,camera_a,camera_b,max_error_px=10, diagnostics=None):
    ca,da=ray(a,camera_a)
    cb,db=ray(b,camera_b)
    c=max(-1.,min(1.,dot(da,db)))
    angle=math.degrees(math.acos(abs(c)))
    if diagnostics is not None:
        diagnostics['ray_angle_deg']=angle
    if angle<2:
        return None,'weak geometry'
    d=diff(cb,ca)
    sa=(dot(d,da)-c*dot(d,db))/(1-c*c)
    sb=(c*dot(d,da)-dot(d,db))/(1-c*c)
    if min(sa,sb)<=0:
        return None,'behind camera'
    point=[(x+sa*u+y+sb*v)/2 for x,u,y,v in zip(ca,da,cb,db)]
    errors=[norm(diff(q,p)) for camera,p in ((camera_a,a),(camera_b,b))
            if (q:=project(point,camera)) is not None]
    if diagnostics is not None and len(errors)==2:
        diagnostics['reprojection_px']=errors
    if len(errors)!=2 or max(errors)>max_error_px:
        return None,'reprojection error'
    return point,sum(errors)/2
def reconstruct(a,b,cameras,confidence=.55,max_error_px=10):
    points,rejected,errors,evidence={},{},[],{}
    for j in JOINTS:
        detail=evidence[j]={'camera_a':a.get(j),'camera_b':b.get(j)}
        if j not in a or j not in b or min(a[j][2],b[j][2])<confidence:
            rejected[j]='low visibility'
            continue
        point,quality=triangulate(a[j][:2],b[j][:2],*cameras,max_error_px,detail)
        if point is None:
            rejected[j]=quality
        else:
            points[j]=point
            errors.append(quality)
    return dict(points=points,rejected=rejected,evidence=evidence,error_px=sum(errors)/len(errors) if errors else None)
def write_joint_csv(frames, path):
    columns=('time_s','source_frame_a','source_frame_b','pair_skew_ms',
             'joint','status','x_m','y_m','z_m','score_a','score_b',
             'ray_angle_deg','reprojection_a_px','reprojection_b_px')
    with Path(path).open('w',encoding='utf-8',newline='') as stream:
        writer=csv.writer(stream)
        writer.writerow(columns)
        for frame in frames:
            for joint,name in JOINTS.items():
                evidence=frame.get('evidence',{}).get(joint,{})
                a,b=(evidence.get(key) for key in ('camera_a','camera_b'))
                point=frame['points'].get(joint)
                writer.writerow([frame['time'],*frame['source_frames'],frame.get('pair_skew_ms'),
                                 name,'reconstructed' if point is not None else frame['rejected'].get(joint,'unavailable'),
                                 *(point if point is not None else (None,)*3),
                                 a[2] if a else None,b[2] if b else None,evidence.get('ray_angle_deg'),
                                 *evidence.get('reprojection_px',(None,None))])

def nearest_frame_index(time_s, fps):
    return math.floor(time_s*fps+.5)

def export_view(frames,title,stem,rate,units='m'):
    if not any(frame['points'] for frame in frames):
        raise ValueError('No landmarks reconstructed. Check synchronization, calibration, visibility, and camera separation.')
    payload=dict(frames=frames,title=title,bones=BONES,joints=JOINTS,rate=rate,units=units)
    data=json.dumps(payload,allow_nan=False)
    write_joint_csv(frames,stem+'_joints.csv')
    Path(stem+'.json').write_text(data,encoding='utf-8')
    html=VIEWER.replace('__DATA__',data.replace('<',r'\u003c'))
    Path(stem+'.html').write_text(html,encoding='utf-8')
    if 'get_ipython' in globals():
        from IPython.display import HTML,display
        display(HTML(f'<iframe title="ReplayAxis viewer" sandbox="allow-scripts" width="100%" height="620" style="border:0" srcdoc="{escape(html,quote=True)}"></iframe>'))
    print(f'Saved {stem}.html {stem}.json and {stem}_joints.csv')

VIEWER = '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>ReplayAxis · Motion Lab</title>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#10171c;color:#ebf3f1;font:15px system-ui}main{max-width:1140px;margin:auto;padding:22px}header{border-top:3px solid #82e6be;padding:14px 0 16px}h1,h2,p{margin:0}h1{font-size:26px}h2{font-size:16px;margin-bottom:14px}p,small{color:#afc2c5}header p{margin-top:5px}.grid{display:grid;grid-template-columns:minmax(0,1fr) 240px;gap:12px}canvas,aside{background:#182329;border:1px solid #354750;border-radius:8px}canvas{background:#071219;display:block;width:100%;height:358px;touch-action:none;cursor:grab}aside{padding:16px}aside select{width:100%;margin:8px 0}aside p{font-size:13px;line-height:1.4}dl{display:grid;gap:8px;margin:0}dt{color:#afc2c5}dd{margin:0 0 7px;font:16px ui-monospace,monospace}.bar{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin:12px 0}select,button{background:#23343d;color:#eef8f4;border:1px solid #45616a;border-radius:6px;padding:8px 12px;cursor:pointer}button:hover,button:focus-visible{border-color:#82e6be}input{flex:1;min-width:110px;accent-color:#82e6be}@media(max-width:650px){.grid{grid-template-columns:1fr}canvas{height:290px}aside{padding:12px}dl{grid-template-columns:auto 1fr}}\n</style>\n<main><header><h1>ReplayAxis</h1><p id="title"></p></header>\n<div class="grid"><canvas tabindex="0" aria-label="3D skeleton. Click a joint to mark it. Drag or use arrow keys to orbit."></canvas><aside><h2>Frame diagnostics</h2><dl><dt>Time</dt><dd id="time"></dd><dt>Source frames · A / B</dt><dd id="source"></dd><dt>Frame-pair skew (B − A)</dt><dd id="skew"></dd><dt>Landmarks reconstructed</dt><dd id="count"></dd><dt>Mean reprojection error</dt><dd id="err"></dd></dl><label for="joint">Inspect joint</label><select id="joint"></select><p id="detail"></p><button id="mark" type="button">Mark joint</button><p id="markers" aria-live="polite"></p></aside></div>\n<div class="bar"><button id="play">Play</button><button id="prev" aria-label="Previous frame">−1 frame</button><button id="next" aria-label="Next frame">+1 frame</button><input type="range" id="scrub" aria-label="Playback frame"><label for="render-mode">View</label><select id="render-mode"><option value="wireframe">Body wireframe</option><option value="skeleton">Skeleton</option></select><button id="reset">Reset view</button></div>\n<small>Wireframe proportions are illustrative. Click a joint to mark it from the current frame. Click it again to unmark. Drag to orbit and scroll to zoom.</small>\n</main>\n<script>\nconst data=__DATA__;\nconst el=id=>document.getElementById(id);\nconst canvas=document.querySelector(\'canvas\'),ctx=canvas.getContext(\'2d\');\nconst frames=data.frames;\nel(\'title\').textContent=data.title;\nconst joint=el(\'joint\');\nfor(const [id,name] of Object.entries(data.joints)){\n const option=document.createElement(\'option\');option.value=id;option.textContent=name;joint.appendChild(option);\n}\njoint.value=\'15\';\njoint.onchange=()=>render();\nconst points=frames.flatMap(f=>Object.values(f.points));\nconst bounds=[0,1,2].map(axis=>points.reduce(([lo,hi],p)=>[Math.min(lo,p[axis]),Math.max(hi,p[axis])],[Infinity,-Infinity]));\nconst center=bounds.map(([lo,hi])=>(lo+hi)/2);\nconst extent=Math.max(.001,...bounds.map(([lo,hi])=>hi-lo));\nlet i=0,yaw=-.44,pitch=.12,zoom=1,drag=null,timer=null;\nconst marks=new Map();\nlet projected={};\nfunction markedAt(id,frame){\n const events=marks.get(id)??[];\n let active=false;\n for(const event of events){\n  if(event.frame>frame)break;\n  active=event.active;\n }\n return active;\n}\nfunction toggleMark(id){\n stop();\n const events=marks.get(id)??[];\n const active=!markedAt(id,i);\n marks.set(id,[...events.filter(event=>event.frame!==i),{frame:i,active}].sort((a,b)=>a.frame-b.frame));\n joint.value=id;\n render();\n}\nconst scrub=el(\'scrub\');scrub.min=0;scrub.max=frames.length-1;scrub.value=0;\nconst add3=(a,b)=>a.map((v,i)=>v+b[i]);\nconst sub3=(a,b)=>a.map((v,i)=>v-b[i]);\nconst mul3=(a,k)=>a.map(v=>v*k);\nconst dot3=(a,b)=>a.reduce((sum,v,i)=>sum+v*b[i],0);\nconst cross3=(a,b)=>[a[1]*b[2]-a[2]*b[1],a[2]*b[0]-a[0]*b[2],a[0]*b[1]-a[1]*b[0]];\nconst unit3=a=>{const length=Math.hypot(...a);return length>1e-7?mul3(a,1/length):null};\nconst mix3=(a,b,t)=>add3(mul3(a,1-t),mul3(b,t));\n\nfunction bodyWireframe(p, bodyScale) {\n const segments=[];\n const appendRingSurface=(origin,end,reference,profile,sides=12)=>{\n  const axis=unit3(sub3(end,origin));\n  if(!axis)return;\n  const crossAxis=ref=>unit3(sub3(ref,mul3(axis,dot3(ref,axis))));\n  const u=crossAxis(reference)??crossAxis(Math.abs(axis[0])<.8?[1,0,0]:[0,1,0]);\n  if(!u)return;\n  const v=cross3(axis,u);\n  let previous=null;\n  for(const [along,wide,deep] of profile){\n   const center=mix3(origin,end,along);\n   const ring=Array.from({length:sides},(_,i)=>{\n    const angle=i*2*Math.PI/sides;\n    return add3(center,add3(mul3(u,wide*Math.cos(angle)),mul3(v,deep*Math.sin(angle))));\n   });\n   for(let j=0;j<sides;j++){\n    segments.push([ring[j],ring[(j+1)%sides]]);\n    if(previous)segments.push([previous[j],ring[j]]);\n   }\n   previous=ring;\n  }\n };\n const shoulder=p[11]&&p[12]?mix3(p[11],p[12],.5):null;\n const hips=p[23]&&p[24]?mix3(p[23],p[24],.5):null;\n const lateral=shoulder?sub3(p[12],p[11]):hips?sub3(p[24],p[23]):[1,0,0];\n const s=bodyScale;\n if(shoulder&&hips){\n  const trunk=Math.hypot(...sub3(shoulder,hips));\n  const shoulderRadius=Math.hypot(...lateral)*.52;\n  const hipRadius=Math.hypot(...sub3(p[24],p[23]))*.62;\n  if(trunk>1e-7)appendRingSurface(hips,shoulder,lateral,[\n   [-.19,hipRadius*.75,.075*s],[-.04,hipRadius,.104*s],\n   [.16,hipRadius*.92,.093*s],[.34,Math.min(hipRadius,shoulderRadius)*.95,.086*s],\n   [.55,shoulderRadius*.75,.103*s],[.74,shoulderRadius*.93,.119*s],\n   [.92,shoulderRadius,.097*s],[1.02,shoulderRadius*.78,.071*s],\n   [1.11,shoulderRadius*.32,.051*s]\n  ],20);\n }\n if(p[0]&&shoulder){\n  const up=unit3(hips?sub3(shoulder,hips):sub3(p[0],shoulder));\n  const side=unit3(lateral);\n  if(up&&side){\n   const forward=cross3(up,side);\n   const headCenter=sub3(p[0],mul3(forward,.041*s));\n   const bottom=sub3(headCenter,mul3(up,.11*s)),top=add3(headCenter,mul3(up,.12*s));\n   appendRingSurface(shoulder,bottom,lateral,[[0,.054*s,.052*s],[.5,.044*s,.048*s],[1,.043*s,.047*s]]);\n   const head=Array.from({length:9},(_,i)=>{\n    const t=i/8, radius=Math.sqrt(Math.max(.003,1-(2*t-1)**2));\n    return [t,.083*s*radius,.095*s*radius];\n   });\n   appendRingSurface(bottom,top,lateral,head,16);\n  }\n }\n const limbs=[[11,13,.068,.048],[12,14,.068,.048],[13,15,.049,.028],[14,16,.049,.028],\n              [23,25,.099,.056],[24,26,.099,.056],[25,27,.059,.032],[26,28,.059,.032]];\n for(const [start,end,r1,r2] of limbs){\n  if(!p[start]||!p[end])continue;\n  const rings=[-.04,0,.16,.32,.5,.68,.84,1,1.04].map(t=>{\n   const radius=s*(r1+(r2-r1)*t)*(1+.11*Math.sin(Math.PI*t));\n   return [t,radius,radius*.86];\n  });\n  appendRingSurface(p[start],p[end],lateral,rings);\n }\n return segments;\n}\n\nconst torsoLengths=frames.filter(({points:p})=>p[11]&&p[12]&&p[23]&&p[24])\n .map(({points:p})=>Math.hypot(...sub3(mix3(p[11],p[12],.5),mix3(p[23],p[24],.5))))\n .filter(length=>length>1e-7).sort((a,b)=>a-b);\nconst bodyScale=torsoLengths.length?torsoLengths[Math.floor(torsoLengths.length/2)]/.62:1;\nel(\'render-mode\').onchange=()=>render();\nfunction render(){\n const w=canvas.clientWidth,h=canvas.clientHeight,dpr=devicePixelRatio||1;\n if(canvas.width!==Math.round(w*dpr)||canvas.height!==Math.round(h*dpr)){canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr)}\n ctx.setTransform(dpr,0,0,dpr,0,0);ctx.clearRect(0,0,w,h);\n ctx.strokeStyle=\'#23353e\';ctx.lineWidth=1;\n for(let x=0;x<w;x+=24){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,h);ctx.stroke()}\n for(let y=0;y<h;y+=24){ctx.beginPath();ctx.moveTo(0,y);ctx.lineTo(w,y);ctx.stroke()}\n const scale=Math.min(w,h)*.76*zoom/(extent+(el(\'render-mode\').value===\'skeleton\'?0:.16*bodyScale));\n function screen(p){\n  const [x,y,z]=p.map((v,j)=>v-center[j]);\n  const a=x*Math.cos(yaw)+z*Math.sin(yaw),b=z*Math.cos(yaw)-x*Math.sin(yaw);\n  return [w/2+a*scale,h/2+(y*Math.cos(pitch)-b*Math.sin(pitch))*scale,y*Math.sin(pitch)+b*Math.cos(pitch)];\n }\n const f=frames[i];\n projected=Object.fromEntries(Object.entries(f.points).map(([j,p])=>[j,screen(p)]));\n ctx.lineCap=\'round\';\n if(el(\'render-mode\').value!==\'skeleton\'){\n  const wires=bodyWireframe(f.points,bodyScale).map(([a,b])=>[screen(a),screen(b)]);\n  const depthRange=Math.max(extent,.01);\n  for(let pass=0;pass<4;pass++){\n   ctx.strokeStyle=\'rgba(40,215,250,\'+(0.2+pass*.19)+\')\';\n   ctx.lineWidth=.72;\n   ctx.beginPath();\n   for(const [a,b] of wires){\n    const depth=(a[2]+b[2])/2;\n    const bucket=Math.max(0,Math.min(3,Math.floor(2-4*depth/depthRange)));\n    if(bucket!==pass)continue;\n    ctx.moveTo(a[0],a[1]);ctx.lineTo(b[0],b[1]);\n   }\n   ctx.stroke();\n  }\n }else{\n  ctx.strokeStyle=\'#39d7f4\';ctx.lineWidth=3;\n  for(const [a,b] of data.bones){\n   if(!projected[a]||!projected[b])continue;\n   ctx.beginPath();ctx.moveTo(projected[a][0],projected[a][1]);ctx.lineTo(projected[b][0],projected[b][1]);ctx.stroke();\n  }\n }\n for(const [id,[x,y]] of Object.entries(projected)){\n  const active=markedAt(id,i);\n  ctx.fillStyle=active?\'#ffd38a\':id===joint.value?\'#b8f6ff\':\'#e9fcff\';\n  ctx.beginPath();ctx.arc(x,y,active?6:id===joint.value?5:4,0,2*Math.PI);ctx.fill();\n  if(active){\n   ctx.beginPath();ctx.arc(x,y,10,0,2*Math.PI);\n   ctx.strokeStyle=\'#ffd38a\';ctx.lineWidth=1.5;ctx.stroke();\n  }\n }\n const detail=f.evidence?.[joint.value]??{};\n const state=f.rejected?.[joint.value]??(f.points[joint.value]?\'Reconstructed\':\'Unavailable\');\n const score=c=>c?c[2].toFixed(2):\'—\';\n const angle=detail.ray_angle_deg==null?\'—\':detail.ray_angle_deg.toFixed(1)+\'°\';\n const residual=detail.reprojection_px?.map(v=>v.toFixed(2)).join(\' / \')??\'—\';\n el(\'detail\').textContent=state+\' · scores A/B \'+score(detail.camera_a)+\' / \'+score(detail.camera_b)+\' · ray angle \'+angle+\' · residual A/B \'+residual+\' px\';\n const visible=Boolean(f.points[joint.value]);\n const isMarked=markedAt(joint.value,i);\n const markButton=el(\'mark\');\n markButton.textContent=isMarked?\'Unmark joint\':\'Mark joint\';\n markButton.disabled=!visible&&!isMarked;\n markButton.setAttribute(\'aria-pressed\',String(isMarked));\n const active=Object.entries(data.joints).filter(([id])=>markedAt(id,i))\n   .map(([id,name])=>f.points[id]?name:name+\' (hidden)\');\n const label=active.length?\'Marked: \'+active.join(\', \'):\'No points marked.\';\n if(el(\'markers\').textContent!==label)el(\'markers\').textContent=label;\n el(\'skew\').textContent=f.pair_skew_ms==null?\'—\':f.pair_skew_ms.toFixed(1)+\' ms\';\n el(\'time\').textContent=f.time.toFixed(3)+\' s\';\n el(\'source\').textContent=f.source_frames.join(\' / \');\n el(\'count\').textContent=Object.keys(projected).length+\' / \'+Object.keys(data.joints).length;\n el(\'err\').textContent=f.error_px===null?\'—\':f.error_px.toFixed(2)+\' px\';\n scrub.value=i;\n}\nfunction show(index){i=Math.max(0,Math.min(frames.length-1,index));render()}\nfunction stop(){clearInterval(timer);timer=null;el(\'play\').textContent=\'Play\'}\nel(\'play\').onclick=()=>{if(timer!==null){stop();return}if(i===frames.length-1)show(0);el(\'play\').textContent=\'Pause\';timer=setInterval(()=>{if(i===frames.length-1)stop();else show(i+1)},1000/data.rate)};\nel(\'prev\').onclick=()=>{stop();show(i-1)};el(\'next\').onclick=()=>{stop();show(i+1)};\nscrub.oninput=()=>{stop();show(Number(scrub.value))};\nel(\'reset\').onclick=()=>{yaw=-.44;pitch=.12;zoom=1;render()};\nel(\'mark\').onclick=()=>toggleMark(joint.value);\ncanvas.onpointerdown=e=>{\n if(!e.isPrimary||e.button!==0)return;\n stop();\n drag={id:e.pointerId,x:e.clientX,y:e.clientY,orbit:false};\n canvas.setPointerCapture(e.pointerId);\n};\ncanvas.onpointermove=e=>{\n if(!drag||drag.id!==e.pointerId)return;\n const dx=e.clientX-drag.x,dy=e.clientY-drag.y;\n if(!drag.orbit&&Math.hypot(dx,dy)<5)return;\n drag.orbit=true;\n yaw+=dx*.012;pitch+=dy*.012;drag.x=e.clientX;drag.y=e.clientY;render();\n};\ncanvas.onpointerup=e=>{\n if(!drag||e.pointerId!==drag.id)return;\n const orbit=drag.orbit||Math.hypot(e.clientX-drag.x,e.clientY-drag.y)>=5;\n drag=null;\n if(orbit)return;\n const rect=canvas.getBoundingClientRect();\n const x=e.clientX-rect.left-canvas.clientLeft,y=e.clientY-rect.top-canvas.clientTop;\n let nearest=null,distance=14;\n for(const [id,[px,py]] of Object.entries(projected)){\n  const d=Math.hypot(px-x,py-y);\n  if(d<distance){nearest=id;distance=d;}\n }\n if(nearest!==null)toggleMark(nearest);\n};\ncanvas.onpointercancel=canvas.onlostpointercapture=()=>drag=null;\ncanvas.addEventListener(\'wheel\',e=>{e.preventDefault();zoom=Math.min(5,Math.max(.25,zoom*Math.exp(-e.deltaY*.001)));render()},{passive:false});\ncanvas.onkeydown=e=>{const key={ArrowLeft:[-.1,0],ArrowRight:[.1,0],ArrowUp:[0,-.1],ArrowDown:[0,.1]}[e.key];if(key){e.preventDefault();yaw+=key[0];pitch+=key[1];render()}};\nnew ResizeObserver(render).observe(canvas);render();\n</script></html>\n'

In [ ]:
identity=[[1,0,0],[0,1,0],[0,0,1]]
k=[[950,0,640],[0,950,360],[0,0,1]]
a,b=[dict(K=k,R=identity,t=t) for t in ([0,0,0],[-1.15,0,0])]
base={0:(0,-.9,0),11:(-.2,-.62,0),12:(.2,-.62,0),13:(-.35,-.36,0),14:(.35,-.36,0),
      15:(-.4,-.18,0),16:(.4,-.18,0),23:(-.13,0,0),24:(.13,0,0),
      25:(-.15,.42,.04),26:(.15,.42,.04),27:(-.16,.81,0),28:(.16,.81,0)}
frames=[]
worst=0
for i in range(120):
    phase=2*math.pi*i/45
    truth={j:[x+.008*i,y,4.6+z+(.22*math.sin(phase+math.pi*(j%2)) if j in (13,14,15,16,25,26,27,28) else 0)]
           for j,(x,y,z) in base.items()}
    first={j:[*project(p,a),1.] for j,p in truth.items()}
    second={j:[*project(p,b),1.] for j,p in truth.items()}
    if 40<=i<57:
        second[15][2]=0.
    f=reconstruct(first,second,(a,b))
    assert len(f['points'])==(12 if 40<=i<57 else 13)
    worst=max(worst,max(norm(diff(p,truth[j])) for j,p in f['points'].items()))
    f.update(time=i/30,source_frames=[i,i],pair_skew_ms=0.)
    frames.append(f)
assert worst<1e-8,worst
assert triangulate([640,360],[640,360],a,b)[1]=='weak geometry'
assert triangulate([640,360],[400,480],a,b)[1]=='reprojection error'
assert [nearest_frame_index(i+.5,1) for i in range(4)]==[1,2,3,4]
assert frames[40]['rejected'][15]=='low visibility'
assert frames[40]['evidence'][15]['camera_b'][2]==0.
assert len(frames[0]['evidence'][11]['reprojection_px'])==2
print(f'Synthetic check passed: {len(frames)} frames, max error {worst:.2e} m, intentional occlusion withheld.')
export_view(frames,'Synthetic motion · known calibration','ReplayAxis_Demo',30)

## Multi-sport fixtures and file input

The next two cells generate and verify five stylized motions: running, freestyle swimming, figure skating, cricket bowling, and a football kick. Each has known 3D coordinates, projected 2D observations, one missing camera sample, one intentionally mistimed pair, and a low-confidence wrist. Both input formats run through the same triangulation as video. Swimming here uses an ideal camera model without water refraction.

The CSV and tab-delimited TXT formats use the same header: sample_id, camera, frame_index, time_s, joint_id, x_px, y_px, score. Supply a shared sample ID and timestamps in seconds on the same clock, MediaPipe joint IDs, and **undistorted** image coordinates in pixels for the supplied stereo calibration. A camera can be absent from a sample. Frames above the configurable 12 ms skew threshold are withheld. This threshold is a starting value, not a validation criterion. File input is offline. The timestamps and camera IDs are the contract a later live adapter must supply.

To open another sport in the viewer, change selected in the last cell. To replay your own landmark file, call reconstruct_landmark_file(path, cameras) and pass the result to export_view. Set the viewer playback rate to the nominal sample rate. The audit retains the actual timestamps. For ordinary video files, use the video section below.

In [ ]:
LANDMARK_FIELDS = ('sample_id', 'camera', 'frame_index', 'time_s', 'joint_id',
                   'x_px', 'y_px', 'score')


def read_landmark_file(path):
    """Read paired, undistorted pixel observations from CSV or tab-delimited text."""
    path = Path(path)
    delimiters = {'.csv': ',', '.tsv': '\t', '.txt': '\t'}
    if path.suffix.lower() not in delimiters:
        raise ValueError('Use CSV, TSV, or tab-delimited TXT with the landmark header.')
    samples = {}
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream, delimiter=delimiters[path.suffix.lower()])
        if reader.fieldnames is None or not set(LANDMARK_FIELDS).issubset(reader.fieldnames):
            raise ValueError(f'Expected columns: {", ".join(LANDMARK_FIELDS)}')
        for line, row in enumerate(reader, start=2):
            try:
                sample, index, joint = (int(row[key]) for key in ('sample_id', 'frame_index', 'joint_id'))
                time, x, y, score = (float(row[key]) for key in ('time_s', 'x_px', 'y_px', 'score'))
                camera = row['camera'].strip().upper()
                if (sample < 0 or index < 0 or time < 0 or joint not in JOINTS or
                    camera not in ('A','B') or not all(map(math.isfinite, (time,x,y,score))) or
                    not 0 <= score <= 1):
                    raise ValueError('invalid camera, frame, joint, time, pixel, or score')
                sample_views = samples.setdefault(sample,{})
                view = sample_views.setdefault(camera, dict(index=index,time=time,points={}))
                if (view['index'],view['time']) != (index,time) or joint in view['points']:
                    raise ValueError('inconsistent frame metadata or duplicate joint')
                view['points'][joint] = [x,y,score]
            except (AttributeError, TypeError, ValueError) as error:
                raise ValueError(f'{path.name}, row {line}: {error}') from error
    if not samples:
        raise ValueError('The landmark file has no samples.')
    return samples


def reconstruct_landmark_file(path, cameras, max_skew_ms=12.):
    """Pair by explicit sample ID and retain missing or high-skew pairs in the audit."""
    if not math.isfinite(max_skew_ms) or max_skew_ms < 0:
        raise ValueError('max_skew_ms must be finite and nonnegative.')
    frames, previous = [], -math.inf
    for sample, views in sorted(read_landmark_file(path).items()):
        frame = reconstruct_sample(sample, views, cameras, max_skew_ms)
        if frame['time'] <= previous:
            raise ValueError(f'Sample timestamps must increase with sample IDs (sample {sample}).')
        previous = frame['time']
        frames.append(frame)
    return frames


def reconstruct_sample(sample, views, cameras, max_skew_ms):
    a,b = (views.get(key) for key in ('A','B'))
    times = [view['time'] if view else None for view in (a,b)]
    indexes = [view['index'] if view else None for view in (a,b)]
    time = sum(view['time'] for view in views.values())/len(views)
    skew = 1000*(b['time']-a['time']) if a and b else None
    reason = ('missing camera' if skew is None else
              'frame-pair skew' if abs(skew)>max_skew_ms else None)
    if reason is None:
        frame = reconstruct(a['points'],b['points'],cameras)
    else:
        frame = dict(points={}, rejected={j:reason for j in JOINTS}, evidence={}, error_px=None)
    frame.update(time=time, source_frames=indexes, source_times=times,
                 pair_skew_ms=skew, sample_id=sample, pair_status=reason or 'paired')
    return frame

In [ ]:
SPORTS = {
    'running': (1.8, .30, .24),
    'freestyle': (.85, .34, .11),
    'figure_skating': (.65, .22, .16),
    'cricket_bowling': (.8, .37, .20),
    'football_kick': (.9, .15, .45),
}


def synthetic_sport_pose(sport, time):
    """Stylized kinematic fixture, with camera A oriented vertically in the image."""
    frequency, arm, leg = SPORTS[sport]
    phase = 2*math.pi*frequency*time
    pose = {}
    for joint,(x,y,z) in base.items():
        side = 1 if joint%2 else -1
        swing = side*math.sin(phase)
        if joint in (13,14,15,16):
            z += arm*swing
            y -= arm*.45*math.cos(phase)
        if joint in (25,26,27,28):
            z += leg*swing*(1.6 if joint in (27,28) else 1)
        if sport=='freestyle':
            y,z = -.24*z,y
        if sport=='figure_skating':
            x,z = x*math.cos(phase)+z*math.sin(phase), z*math.cos(phase)-x*math.sin(phase)
        if sport=='cricket_bowling' and joint in (14,16):
            y -= (joint==16)*.25 + .65*max(0,math.sin(phase))
        if sport=='football_kick' and joint in (26,28):
            z += .35*max(0,math.sin(phase))
        pose[joint] = [x+.14*time, y, z+4.7]
    return pose


def run_sports_fixtures(directory='ReplayAxis_Sports', selected='running'):
    directory = Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    if selected not in SPORTS:
        raise ValueError(f'Choose from {", ".join(SPORTS)}.')
    results = []
    cameras = (a,b)
    for sport in SPORTS:
        observations, truth = [], []
        for i in range(48):
            time = i/30
            pose = synthetic_sport_pose(sport,time)
            truth.append(pose)
            for camera_name,camera in zip(('A','B'),cameras):
                if i==24 and camera_name=='B':
                    continue
                for joint,point in pose.items():
                    x,y = project(point,camera)
                    score = 0. if 12<=i<16 and joint==15 and camera_name=='B' else 1.
                    offset = .024 if i==32 and camera_name=='B' else 0.
                    observations.append((i,camera_name,i,time+offset,joint,x,y,score))
        csv_path = directory/(sport+'.csv')
        text_path = directory/(sport+'.txt')
        for path,delimiter in ((csv_path,','),(text_path,'\t')):
            with path.open('w',newline='',encoding='utf-8') as stream:
                writer = csv.writer(stream,delimiter=delimiter)
                writer.writerow(LANDMARK_FIELDS)
                writer.writerows(observations)
        replay = reconstruct_landmark_file(csv_path,cameras)
        assert replay==reconstruct_landmark_file(text_path,cameras)
        assert replay[24]['pair_status']=='missing camera' and replay[32]['pair_status']=='frame-pair skew'
        assert replay[12]['rejected'][15]=='low visibility'
        error = max(norm(diff(point,truth[i][joint])) for i,frame in enumerate(replay)
                    for joint,point in frame['points'].items())
        assert error<1e-8
        coverage = sum(len(frame['points']) for frame in replay)/(len(replay)*len(JOINTS))
        results.append(dict(sport=sport,frames=len(replay),joint_coverage=coverage,max_error_m=error))
        if sport==selected:
            export_view(replay,f'{sport.replace("_"," ").title()} · synthetic',str(directory/sport),30)
    (directory/'calibration.json').write_text(json.dumps(dict(cameras=cameras,units='m'),indent=2),encoding='utf-8')
    with (directory/'summary.csv').open('w',newline='',encoding='utf-8') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    return results


for result in run_sports_fixtures():
    print(result)

## Rehearse incremental input

A future capture adapter calls StreamingStereo.push(sample_id, camera, view) once per camera per shared sample. A view has index, time in seconds on a shared clock, and points mapping joint IDs to undistorted [x, y, score] values. Each push returns finalized frames. The default four-sample reorder window lets packets arrive out of order. A pair expires when the largest observed sample ID is more than four ahead, or when finish() is called. Duplicate or late packets are counted. When both cameras are absent, no sample exists to report. The window advances from incoming sample IDs. A stalled source requires an upstream timeout or end-of-stream signal. This rehearsal exercises the reconstruction path incrementally using recorded observations. Hardware capture and latency still need measurement.

In [ ]:
class StreamingStereo:
    """Pair complete camera-detection packets sharing a sample ID and clock."""

    def __init__(self, cameras, *, reorder_samples=4, max_skew_ms=12.):
        if not isinstance(reorder_samples, int) or reorder_samples < 0:
            raise ValueError('reorder_samples must be a nonnegative integer.')
        if not math.isfinite(max_skew_ms) or max_skew_ms < 0:
            raise ValueError('max_skew_ms must be finite and nonnegative.')
        self.cameras = cameras
        self.reorder_samples = reorder_samples
        self.max_skew_ms = max_skew_ms
        self.pending = {}
        self.highest_sample = -1
        self.closed_through = -1
        self.late_packets = 0
        self.duplicate_packets = 0
        self.finished = False

    def push(self, sample, camera, view):
        """Accept one complete view; return newly finalized 3D frames."""
        if self.finished:
            raise RuntimeError('This stream has already been closed.')
        if not isinstance(sample, int) or sample < 0 or camera not in ('A','B'):
            raise ValueError('Expected a nonnegative integer sample and camera A or B.')
        if sample <= self.closed_through:
            self.late_packets += 1
            return []
        views = self.pending.setdefault(sample,{})
        if camera in views:
            self.duplicate_packets += 1
            return []
        views[camera] = view
        self.highest_sample = max(self.highest_sample,sample)
        return self._finalize(self.highest_sample-self.reorder_samples-1)

    def _finalize(self, through):
        frames = []
        for sample in sorted(self.pending):
            if sample > through:
                break
            views = self.pending.pop(sample)
            frames.append(reconstruct_sample(sample,views,self.cameras,self.max_skew_ms))
        self.closed_through = max(self.closed_through,through)
        return frames

    def finish(self):
        """Finalize pending samples once at the end of a capture."""
        self.finished = True
        return self._finalize(self.highest_sample)


def rehearse_stream(path, cameras):
    """Replay recorded packets in deterministic arrival order, without a camera."""
    samples = read_landmark_file(path)
    stream = StreamingStereo(cameras)
    packets = []
    for sample, views in sorted(samples.items()):
        for camera, view in views.items():
            delay = 2 if camera=='B' and sample%3==0 else 0
            packets.append((sample+delay,sample,camera,view))
    results = []
    for _,sample,camera,view in sorted(packets):
        results.extend(stream.push(sample,camera,view))
    return results+stream.finish(),stream

In [ ]:
for sport in SPORTS:
    source = Path('ReplayAxis_Sports')/(sport+'.csv')
    streamed, stream = rehearse_stream(source,(a,b))
    assert streamed==reconstruct_landmark_file(source,(a,b))
    assert stream.late_packets==stream.duplicate_packets==0
    print(f'{sport}: {len(streamed)} streamed samples match offline reconstruction.')

# Send a duplicate, then deliver the second camera after the pair has expired.
example = read_landmark_file(Path('ReplayAxis_Sports/running.csv'))
stream = StreamingStereo((a,b),reorder_samples=1)
assert stream.push(0,'A',example[0]['A'])==[]
assert stream.push(0,'A',example[0]['A'])==[]
assert stream.push(2,'A',example[2]['A'])[0]['pair_status']=='missing camera'
assert stream.push(0,'B',example[0]['B'])==[]
assert stream.duplicate_packets==stream.late_packets==1
assert stream.finish()[0]['sample_id']==2
assert stream.finish()==[]
print('Late and duplicate packets are counted, and finalized frames are stable.')

### Video playback timing rehearsal

Two file readers feed the same incremental stereo pairing used above. In paced mode the reader targets video playback speed. If processing falls behind, the run keeps every available sample and records the deadline lag. The timing report separates skew rejection, missing views, and samples that were never emitted after both streams ended. Processing measurements include frame decoding, pose inference, and incremental reconstruction. They exclude preparation, final report export, and browser rendering. The next test uses mock readers and a virtual clock.

In [ ]:
import time


def replay_video_pair(read_view, fps, starts, duration, rate, cameras, *,
                      paced=False, max_skew_ms=12., clock=time.perf_counter, sleep=time.sleep):
    """Feed a pair of timed readers through stereo pairing and measure processing."""
    rate = min(rate,*fps)
    stream = StreamingStereo(cameras,reorder_samples=0,max_skew_ms=max_skew_ms)
    frames, processing, lag = [], [], []
    active = [True,True]
    start_clock = clock()
    requested = math.ceil(duration*rate)
    last_tick = None
    for tick in range(requested):
        scheduled = start_clock+tick/rate
        if paced:
            remaining = scheduled-clock()
            if remaining>0:
                sleep(remaining)
        before = clock()
        received = False
        for slot,camera in enumerate(('A','B')):
            if not active[slot]:
                continue
            index = nearest_frame_index(starts[slot]+tick/rate,fps[slot])
            points = read_view(slot,index)
            if points is None:
                active[slot] = False
                continue
            received = True
            frames.extend(stream.push(tick,camera,dict(index=index,
                          time=index/fps[slot]-starts[slot],points=points)))
        if not received:
            break
        after = clock()
        processing.append((after-before)*1000)
        if paced:
            lag.append(max(0.,after-scheduled-1/rate)*1000)
        last_tick = tick
    frames.extend(stream.finish())
    for frame in frames:
        frame['aligned_source_times'] = frame['source_times']
        frame['source_times'] = [index/f if index is not None else None
                                 for index,f in zip(frame['source_frames'],fps)]
        frame['time'] = frame['sample_id']/rate
    budget = 1000/rate
    report = dict(mode='paced' if paced else 'unpaced', sample_rate_hz=rate,
                  requested_samples=requested, emitted_samples=len(frames),
                  remaining_samples_after_eof=requested-1-last_tick if last_tick is not None else requested,
                  paired_samples=sum(f['pair_status']=='paired' for f in frames),
                  skew_rejected_samples=sum(f['pair_status']=='frame-pair skew' for f in frames),
                  missing_view_samples=sum(f['pair_status']=='missing camera' for f in frames),
                  reconstructed_samples=sum(bool(f['points']) for f in frames),
                  per_sample_budget_ms=budget,
                  mean_processing_ms=sum(processing)/len(processing) if processing else None,
                  max_processing_ms=max(processing,default=None),
                  over_budget_samples=sum(ms>budget for ms in processing),
                  max_processing_deadline_lag_ms=max(lag,default=None),
                  elapsed_ms=(clock()-start_clock)*1000)
    return frames,report

In [ ]:
# Use a virtual clock and known pose detections to exercise video timing without dependencies.
virtual_time=[0.]
def advance(seconds):
    virtual_time[0]+=seconds
def fake_read(slot,index):
    advance(.003)
    if slot==1 and index>=4:
        return None
    return {j:[*project([x,y,z+4.7],(a,b)[slot]),1.] for j,(x,y,z) in base.items()}

mock_frames,timing = replay_video_pair(fake_read,(30,30),(0,0),.2,30,(a,b),
                                      paced=True,clock=lambda:virtual_time[0],sleep=advance)
assert timing['paired_samples']==4 and timing['missing_view_samples']==2
assert timing['over_budget_samples']==0 and timing['max_processing_deadline_lag_ms']==0
assert mock_frames[-1]['source_frames']==[5,None]
assert set(mock_frames[-1]['rejected'].values())=={'missing camera'}

virtual_time[0]=0
def slow_read(slot,index):
    advance(.023)
    return fake_read(slot,index)
_, slow_timing = replay_video_pair(slow_read,(30,30),(0,0),.1,30,(a,b),
                                   paced=True,clock=lambda:virtual_time[0],sleep=advance)
assert slow_timing['over_budget_samples']==3
assert slow_timing['max_processing_deadline_lag_ms']>0
print('Video timing rehearsal: intact pairs, early EOF, and pacing overload handled.')

## Verify an independent distance

Use two identifiable endpoints on a rigid target with a measured separation. Observe it in several positions and orientations within the intended capture volume, on frames held out of calibration. Enter raw pixel coordinates for the same endpoint in both cameras. The tolerance is an example, so set a task-specific tolerance before inspecting results. This tests geometric scale and correspondence. Anatomical accuracy requires an independent biomechanical reference.

In [ ]:
def summarize_distances(records, reference_m, tolerance_mm):
    """Compare visible endpoint pairs with an independently measured distance."""
    if not math.isfinite(reference_m) or reference_m <= 0:
        raise ValueError('Reference distance must be a positive number in metres.')
    if not math.isfinite(tolerance_mm) or tolerance_mm <= 0:
        raise ValueError('Tolerance must be a positive number in millimetres.')
    rows = []
    for label, first, second, reason in records:
        distance = norm(diff(first, second)) if first is not None and second is not None else None
        error = None if distance is None else 1000*(distance-reference_m)
        rows.append(dict(sample=label, status=reason or 'measured', distance_m=distance,
                         signed_error_mm=error, within_tolerance=None if error is None
                         else abs(error)<=tolerance_mm))
    errors = [r['signed_error_mm'] for r in rows if r['signed_error_mm'] is not None]
    count = len(errors)
    stats = dict(samples=len(rows), measured=count, coverage=count/len(rows) if rows else None,
                 bias_mm=sum(errors)/count if count else None,
                 mae_mm=sum(abs(e) for e in errors)/count if count else None,
                 max_error_mm=max(map(abs,errors)) if count else None,
                 within_tolerance=sum(bool(r['within_tolerance']) for r in rows),
                 reference_m=reference_m, tolerance_mm=tolerance_mm)
    return stats, rows


def validate_rigid_target(samples, cameras, reference_m, tolerance_mm, *, correct_pixel=None):
    """Each sample holds matching endpoint pixels in camera order A then B."""
    records = []
    for sample in samples:
        points, failures = [], []
        for endpoint in ('first','second'):
            pair = sample.get(endpoint)
            if pair is None or any(pixel is None for pixel in pair):
                points.append(None)
                failures.append(f'{endpoint}: missing observation')
                continue
            if len(pair)!=2 or any(len(pixel)!=2 or not all(map(math.isfinite,pixel)) for pixel in pair):
                raise ValueError(f'{sample["sample"]}: expected two finite [x, y] pixels for {endpoint}.')
            if correct_pixel is not None:
                pair = [correct_pixel(pixel, camera) for pixel,camera in zip(pair,cameras)]
            point, diagnostic = triangulate(*pair,*cameras)
            points.append(point)
            if point is None:
                failures.append(f'{endpoint}: {diagnostic}')
        records.append((sample['sample'],*points,'; '.join(failures)))
    return summarize_distances(records,reference_m,tolerance_mm)


def write_distance_report(stats, rows, stem='ReplayAxis_Distance_Check'):
    Path(stem+'.json').write_text(json.dumps(stats,indent=2,allow_nan=False),encoding='utf-8')
    with Path(stem+'.csv').open('w',encoding='utf-8',newline='') as stream:
        fields=('sample','status','distance_m','signed_error_mm','within_tolerance')
        writer=csv.DictWriter(stream,fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(json.dumps(stats,indent=2,allow_nan=False))

In [ ]:
# Independent rigid segment with a known 0.5 m separation.
cameras=(a,b)
targets=[]
for label, point in [('near',[-.2,.1,3.2]),('middle',[.2,.1,4.6]),('far',[.5,-.1,6.1])]:
    tip=[point[0]+.5,*point[1:]]
    targets.append(dict(sample=label,first=[project(point,c) for c in cameras],
                        second=[project(tip,c) for c in cameras]))
missing=dict(sample='occluded',first=targets[0]['first'],second=None)
stats,rows=validate_rigid_target([*targets,missing],cameras,.5,5)
assert stats['measured']==3 and stats['coverage']==.75 and stats['mae_mm']<1e-7
assert rows[-1]['distance_m'] is None
# A wrong horizontal match can still have near-zero reprojection error in rectified stereo.
bad_tip=[targets[1]['second'][0], [targets[1]['second'][1][0]+24,targets[1]['second'][1][1]]]
_, residual=triangulate(*bad_tip,*cameras)
bad_stats,_=validate_rigid_target([dict(targets[1],second=bad_tip)],cameras,.5,5)
assert residual<1e-7 and bad_stats['within_tolerance']==0 and bad_stats['mae_mm']>5
write_distance_report(stats,rows,'ReplayAxis_Synthetic_Distance_Check')
print(f'Wrong correspondence: {residual:.2e} px residual, {bad_stats["mae_mm"]:.1f} mm distance error.')

## Analyze your footage

Use a Python 3.11 or 3.12 Jupyter kernel for the optional real-video cells. Install the packages below once, then restart the kernel and run the core cell again. OpenCV decodes videos and corrects lens distortion. MediaPipe estimates image landmarks and ReplayAxis triangulates matched landmarks. The model downloads on first use. You can also place Google's pose_landmarker_lite.task file beside the notebook.

If MediaPipe cannot initialize, select the optional OpenCV CPU backend by setting ONNX_MODEL below to the path of a fixed 640×640 YOLOv8n-pose ONNX export with output shape (1, 56, 8400). The [tested model export](https://huggingface.co/Xenova/yolov8-pose-onnx/blob/main/yolov8n-pose.onnx) has SHA-256 04f6d2416266f2aba6c5ba8b26de33ed9eba3279f972ac02e33f9f1366547586. Obtain model weights separately and review their terms for your use. For this backend, install OpenCV and NumPy. MediaPipe is optional. Both backends assume one athlete in each view. The ONNX backend selects the highest-scoring person per frame.

Use constant-frame-rate clips. Set matching start times from a common event, such as a visible clap or flash, or use synchronized camera timestamps. This pipeline selects the nearest frame in each view. Fast motion requires tight synchronization, and start offsets alone do not correct clock drift. Source times are included in the JSON for auditing.

In [ ]:
# Uncomment to install in the current Jupyter kernel:
# %pip install "mediapipe>=0.10,<0.11" "opencv-contrib-python>=4.8,<5" "numpy>=1.24,<3"

In [ ]:
import contextlib
import urllib.request
import cv2
import numpy as np
MODEL=Path('pose_landmarker_lite.task')
MODEL_URL=('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
           'pose_landmarker_lite/float16/1/pose_landmarker_lite.task')

def correct_pose_pixels(observations, camera):
    if not observations:
        return {}
    joints=list(observations)
    pixels=np.array([[observations[j][:2]] for j in joints],dtype=float)
    k=np.asarray(camera['K'],dtype=float)
    lens=np.asarray(camera.get('dist',[0,0,0,0,0]),dtype=float)
    corrected=cv2.undistortPoints(pixels,k,lens,P=k).reshape(-1,2)
    width,height=camera['image_size']
    return {j:[float(x),float(y),observations[j][2]]
            for j,(x,y),(raw_x,raw_y) in zip(joints,corrected,pixels[:,0])
            if math.isfinite(x) and math.isfinite(y) and 0<=raw_x<width and 0<=raw_y<height}


def detect_pose(detector, image, frame_index, fps):
    import mediapipe as mp
    rgb=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    height,width=rgb.shape[:2]
    result=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb),
                                     round(frame_index*1000/fps))
    if not result.pose_landmarks:
        return {}
    pose=result.pose_landmarks[0]
    return {j:[pose[j].x*width,pose[j].y*height,min(pose[j].visibility,pose[j].presence)]
            for j in JOINTS}


def detect_onnx_pose(net, image):
    """Run the fixed 640-square YOLOv8n-pose export on a single-person view."""
    height,width=image.shape[:2]
    scale=640/max(height,width)
    w,h=round(width*scale),round(height*scale)
    left,top=(640-w)//2,(640-h)//2
    canvas=np.full((640,640,3),114,dtype=np.uint8)
    canvas[top:top+h,left:left+w]=cv2.resize(image,(w,h))
    net.setInput(cv2.dnn.blobFromImage(canvas,1/255,swapRB=True))
    prediction=net.forward()
    if prediction.shape!=(1,56,8400):
        raise ValueError(f'Expected fixed YOLOv8n-pose output (1, 56, 8400), got {prediction.shape}.')
    candidates=prediction[0]
    best=candidates[:,int(np.argmax(candidates[4]))]
    if not np.isfinite(best[4]) or best[4]<.25:
        return {}
    # COCO pose indexes mapped to the ReplayAxis MediaPipe IDs.
    mapping=(0,11,12,13,14,15,16,23,24,25,26,27,28)
    return {joint:[float((x-left)*width/w),float((y-top)*height/h),float(score)]
            for joint,(x,y,score) in zip(mapping,best[5:].reshape(17,3)[[0,*range(5,17)]])
            if all(map(math.isfinite,(x,y,score))) and 0<=score<=1}


def analyze(video_a,video_b,calibration,start_a=0.,start_b=0.,duration=5.,rate=20., *, paced=False, max_skew_ms=12., onnx_model=None):
    if not all(math.isfinite(v) for v in (start_a,start_b,duration,rate)) or min(start_a,start_b)<0 or min(duration,rate)<=0:
        raise ValueError('Start times must be nonnegative and duration and rate positive.')
    cameras=json.loads(Path(calibration).read_text())['cameras']
    if len(cameras)!=2:
        raise ValueError('Calibration must contain exactly two cameras in video order.')
    for camera in cameras:
        k,r,t=(np.asarray(camera[key],dtype=float) for key in ('K','R','t'))
        if (k.shape!=(3,3) or r.shape!=(3,3) or t.shape!=(3,) or
            not all(np.isfinite(item).all() for item in (k,r,t)) or
            min(k[0,0],k[1,1])<=0 or abs(k[0,1])>1e-8 or
            not np.allclose(k[2],[0,0,1]) or not np.allclose(r@r.T,np.eye(3),atol=1e-4)
            or not np.isclose(np.linalg.det(r),1,atol=1e-4)):
            raise ValueError('Invalid calibration: expected zero-skew K, proper rotation R and translation t.')
    centers=[ray([0,0],camera)[0] for camera in cameras]
    if norm(diff(*centers))<1e-6:
        raise ValueError('Camera centers must be distinct.')
    if onnx_model is None:
        import mediapipe as mp
        if not MODEL.exists():
            partial=MODEL.with_suffix('.download')
            try:
                urllib.request.urlretrieve(MODEL_URL,partial)
                partial.replace(MODEL)
            finally:
                partial.unlink(missing_ok=True)
        options=mp.tasks.vision.PoseLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,num_poses=1)
    else:
        net=cv2.dnn.readNetFromONNX(str(onnx_model))
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    with contextlib.ExitStack() as stack:
        captures=[]
        for path,camera in zip((video_a,video_b),cameras):
            capture=cv2.VideoCapture(str(path))
            stack.callback(capture.release)
            if not capture.isOpened():
                raise ValueError(f'Cannot open video: {path}')
            size=[int(capture.get(prop)) for prop in (cv2.CAP_PROP_FRAME_WIDTH,cv2.CAP_PROP_FRAME_HEIGHT)]
            if size!=camera['image_size']:
                raise ValueError(f'{path}: size {size} does not match calibration {camera["image_size"]}.')
            captures.append(capture)
        fps=[c.get(cv2.CAP_PROP_FPS) for c in captures]
        if not all(math.isfinite(f) and 0<f<1000 for f in fps):
            raise ValueError('Use readable constant-frame-rate sources with valid FPS metadata.')
        rate=min(rate,*fps)
        if onnx_model is None:
            detectors=[stack.enter_context(mp.tasks.vision.PoseLandmarker.create_from_options(options)) for _ in captures]
        last=[-1,-1]
        cached=[None,None]
        def read_view(slot,index):
            if index==last[slot]:
                return cached[slot]
            cap=captures[slot]
            while last[slot]<index-1:
                if not cap.grab():
                    return None
                last[slot]+=1
            ok,img=cap.read()
            if not ok:
                return None
            last[slot]=index
            raw=detect_pose(detectors[slot],img,index,fps[slot]) if onnx_model is None else detect_onnx_pose(net,img)
            cached[slot]=correct_pose_pixels(raw,cameras[slot])
            return cached[slot]

        frames,report=replay_video_pair(read_view,fps,(start_a,start_b),duration,rate,cameras,
                                        paced=paced,max_skew_ms=max_skew_ms)
    report['pose_backend']='MediaPipe' if onnx_model is None else 'OpenCV DNN / YOLOv8n-pose'
    Path('ReplayAxis_Real_Clip_timing.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(json.dumps(report,indent=2))
    export_view(frames,'Two-view reconstruction · calibrated','ReplayAxis_Real_Clip',report['sample_rate_hz'])
    return frames

### Optional checkerboard calibration

If you already have a calibration, skip this section. Otherwise capture at least ten paired stills of a checkerboard, in varied positions and orientations throughout the capture volume. Keep the camera settings and placement identical to the videos, and keep the board still during each pair. Put the files in calibration_A and calibration_B with matching filenames. Specify internal corners (columns, rows) and measured square size in metres. Use an asymmetric placement or mark the board outside the corner grid so corner ordering can be checked in both views.

The JSON requires an ordered cameras list. Each entry contains K (3×3 intrinsics, zero skew), R (3×3 world-to-camera rotation), t (three world-to-camera translation values in metres), image_size ([width, height]) and optional dist (OpenCV distortion coefficients). Both transforms must use the same world coordinate system. Verify the calibration with a known distance that was not used in the fit.

In [ ]:
def calibrate_checkerboard(folder_a,folder_b,inner_corners=(9,6),square_m=.025,
                           output='stereo_calibration.json'):
    if min(inner_corners)<3 or square_m<=0:
        raise ValueError('Specify at least 3 × 3 inner corners and a positive square size in metres.')
    board=np.zeros((inner_corners[0]*inner_corners[1],3),np.float32)
    board[:,:2]=np.mgrid[0:inner_corners[0],0:inner_corners[1]].T.reshape(-1,2)*square_m
    points,first,second=[],[],[]
    image_size=None
    for path in sorted(Path(folder_a).iterdir()):
        other=Path(folder_b)/path.name
        if not path.is_file() or not other.is_file():
            continue
        a,b=(cv2.imread(str(p),cv2.IMREAD_GRAYSCALE) for p in (path,other))
        if a is None or b is None:
            continue
        size=a.shape[::-1]
        if a.shape!=b.shape or (image_size is not None and size!=image_size):
            raise ValueError('This calibration helper requires equal image dimensions in both cameras and all pairs.')
        ok_a,ca=cv2.findChessboardCornersSB(a,inner_corners)
        ok_b,cb=cv2.findChessboardCornersSB(b,inner_corners)
        if ok_a and ok_b:
            image_size=size
            points.append(board.copy())
            first.append(ca)
            second.append(cb)
    if len(points)<10:
        raise ValueError(f'Only {len(points)} valid pairs. Provide at least 10 varied checkerboard poses.')
    rms_a,ka,da,*_=cv2.calibrateCamera(points,first,image_size,None,None)
    rms_b,kb,db,*_=cv2.calibrateCamera(points,second,image_size,None,None)
    rms,ka,da,kb,db,r,t,*_=cv2.stereoCalibrate(points,first,second,ka,da,kb,db,
                                                image_size,flags=cv2.CALIB_FIX_INTRINSIC)
    cameras=[dict(K=k.tolist(),dist=d.ravel().tolist(),R=rotation.tolist(),
                  t=translation.ravel().tolist(),image_size=list(image_size))
             for k,d,rotation,translation in [(ka,da,np.eye(3),np.zeros(3)),(kb,db,r,t)]]
    data=dict(cameras=cameras,units='m',pairs=len(points),stereo_rms_px=float(rms))
    Path(output).write_text(json.dumps(data,indent=2))
    print(f'{len(points)} pairs · camera RMS {rms_a:.2f}, {rms_b:.2f} px · stereo RMS {rms:.2f} px')
    if rms>1:
        print('Review image pairs and corner ordering before analysis. Independent geometry validation is still needed.')
    return data

# calibrate_checkerboard('calibration_A','calibration_B',inner_corners=(9,6),square_m=.025)

### Capture preflight

Use map_camera_to_video only after verifying crop origin and dimensions from the acquisition or export workflow. Width differences alone cannot identify the crop. Rotation, mirroring and projective warps require a separate calibration or explicit mapping. A verified crop and resize changes the intrinsics and keeps the lens distortion coefficients and extrinsics. Save the adapted cameras in a new JSON file and point CALIBRATION at it.

For timing, identify the first frame showing each shared flash or other visible event in both cameras. Use audit_sync_events([(frame_a_1,frame_b_1),(frame_a_2,frame_b_2)], fps_a, fps_b). Positive offset means the event appears at a later timestamp in B. Compare the observed spread and frame quantization with your task tolerance before using the estimate. The notebook reports the offset and leaves it unapplied. Configure START_A_SECONDS and START_B_SECONDS from one verified event or external synchronization information. Use events near both ends of the usable clip to assess whether a constant offset is plausible.

In [ ]:
def map_camera_to_video(camera, crop, video_size):
    """Map OpenCV intrinsics through a verified crop and ordinary image resize."""
    source_width,source_height=camera['image_size']
    left,top,width,height=crop
    out_width,out_height=video_size
    if (not all(isinstance(n,int) for n in (*crop,*video_size)) or
        min(left,top)<0 or min(width,height,out_width,out_height)<=0 or
        left+width>source_width or top+height>source_height):
        raise ValueError('Use an in-bounds integer crop and positive video dimensions.')
    sx,sy=out_width/width,out_height/height
    k=camera['K']
    result=dict(camera)
    # OpenCV resize maps pixel centres via (x + 0.5) * scale - 0.5.
    result['K']=[
        [sx*k[0][0],sx*k[0][1],sx*(k[0][2]-left+.5)-.5],
        [sy*k[1][0],sy*k[1][1],sy*(k[1][2]-top+.5)-.5],
        [0.,0.,1.],
    ]
    result['image_size']=list(video_size)
    result['image_mapping']=dict(calibration_size=list(camera['image_size']),crop=list(crop),
                                 resize_to=list(video_size))
    return result


def audit_sync_events(events, fps_a, fps_b, *, max_spread_ms=12.):
    """Use independently identified shared events, specified as zero-based frame pairs."""
    from statistics import median
    if not all(math.isfinite(v) and v>0 for v in (fps_a,fps_b,max_spread_ms)):
        raise ValueError('Frame rates and spread limit must be finite and positive.')
    pairs=list(events)
    if len(pairs)<2:
        raise ValueError('Provide at least two shared events, preferably near the start and end.')
    if any(len(pair)!=2 or any(not isinstance(i,int) or i<0 for i in pair) for pair in pairs):
        raise ValueError('Events must be nonnegative integer frame pairs.')
    if any(a2<=a1 or b2<=b1 for (a1,b1),(a2,b2) in zip(pairs,pairs[1:])):
        raise ValueError('Events must increase in both camera streams.')
    offsets=[b/fps_b-a/fps_a for a,b in pairs]
    estimate=median(offsets)
    spread=(max(offsets)-min(offsets))*1000
    # First-frame-per-event tagging puts each time within one source-frame interval.
    # The sum is a conservative bound for the observed offset spread due to quantization.
    quantization_spread_ms=1000*(1/fps_a+1/fps_b)
    return dict(
        events=len(pairs), offset_b_minus_a_s=estimate,
        offsets_ms=[1000*x for x in offsets], observed_spread_ms=spread,
        frame_quantization_spread_ms=quantization_spread_ms,
        max_spread_ms=max_spread_ms, consistent_with_limit=spread<=max_spread_ms+1e-9,
        resolution_coarser_than_limit=quantization_spread_ms>max_spread_ms,
    )

# Synthetic checks for a noncentral crop, anisotropic resize, and event alignment.
_test_camera=dict(K=[[800,0,543.5],[0,900,959.5],[0,0,1]],image_size=[1088,1920],
                  R=[[1,0,0],[0,1,0],[0,0,1]],t=[0,0,0])
_mapped=map_camera_to_video(_test_camera,(8,12,1080,1896),(540,948))
assert _mapped['K'][0]==[400.,0.,267.5] and _mapped['K'][1]==[0.,450.,473.5]
assert _test_camera['image_size']==[1088,1920]
assert math.isclose(audit_sync_events([(30,33),(120,123)],60,60)['offset_b_minus_a_s'],.05)
assert not audit_sync_events([(30,33),(120,126)],60,60)['consistent_with_limit']
print('Calibration crop/resize and shared-event timing checks passed.')

### Run the analysis

Set the paths and matching times below. For example, if a clap occurs at 0.60 s in angle A and 1.05 s in angle B, enter those two values. Start with a short segment. Keep a single athlete in both views and check that landmarks belong to the same person. The notebook withholds low-visibility joints, weak ray intersections and high-reprojection-error estimates. Reprojection error helps diagnose geometry and does not measure real-world anatomical accuracy.

In [ ]:
VIDEO_A = Path('angle_A.mp4')
VIDEO_B = Path('angle_B.mp4')
CALIBRATION = Path('stereo_calibration.json')
START_A_SECONDS = 0.0
START_B_SECONDS = 0.0
DURATION_SECONDS = 5.0
SAMPLE_HZ = 20.0
ONNX_MODEL = None  # Or Path('yolov8n-pose.onnx') for OpenCV CPU inference.

# frames = analyze(VIDEO_A, VIDEO_B, CALIBRATION, START_A_SECONDS, START_B_SECONDS,
#                  DURATION_SECONDS, SAMPLE_HZ, paced=True, max_skew_ms=12., onnx_model=ONNX_MODEL)

### Run the distance check on your capture

Open [ReplayAxis_Endpoint_Picker.html](ReplayAxis_Endpoint_Picker.html) from the repository root in your browser. Load matching stills at the calibrated resolution, click the two endpoints in the same order, add each sample, and download validation_points.json beside this notebook. You can also create the JSON manually with one entry per held-out pose. Each entry has a sample label and the two endpoints. For example: {"sample":"centre-01", "first":[[420,245],[318,245]], "second":[[525,270],[423,270]]}. Coordinates within each endpoint are ordered Camera A then Camera B. Use null for a missing endpoint observation. Measure the target length independently and replace the example reference below. This cell uses the same calibration, including lens correction, as the real-video reconstruction.

In [ ]:
def correct_validation_pixel(pixel, camera):
    k=np.asarray(camera['K'],dtype=float)
    distortion=np.asarray(camera.get('dist',[0,0,0,0,0]),dtype=float)
    return cv2.undistortPoints(np.asarray(pixel,dtype=float).reshape(1,1,2),
                             k,distortion,P=k).reshape(2).tolist()

# cameras = json.loads(CALIBRATION.read_text())['cameras']
# samples = json.loads(Path('validation_points.json').read_text())
# stats, rows = validate_rigid_target(samples, cameras, reference_m=.50,
#                                     tolerance_mm=5., correct_pixel=correct_validation_pixel)
# write_distance_report(stats, rows)